# Capstone — Your book recommender

The town library has asked for one thing: **ten book suggestions for every reader, on the front page of the
catalog.**
Not twelve experiments. One system, running, with a reason behind every piece of it.

Over thirteen units you built every piece you need: popularity, keyword (BM25) search, item-item neighbours,
matrix factorization, word embeddings, a two-tower model served through an HNSW index, a feature tower that can
place a book nobody has read yet, a sparse + dense hybrid, calibrated blends, a learned reranker, and the tools
to ask *who* a recommender serves.
Unit 14 assembled all of it once.
Now it is your turn to build **your** recommender, measure it honestly, and defend it to the library.

**This is not a race to the highest hit@10.**
In Unit 14 more machinery did *not* automatically win.
The systems with the best hit were the narrowest; the system with the widest reach was the oldest.
A real team has to pick a point on the **hit ↔ reach ↔ exposure frontier** and say why.
Your final deliverable is that choice and its defence, backed by your own tables.

**Rules of the build.**
- Everything is measured on the **`val` split** of the main log at `K = 10`, over the 500 warm readers of the val
  cohort. The sealed final split stays sealed: it was opened once at each checkpoint, and never again.
- The expensive models (MF, the two-tower, the feature towers, the HNSW index, the rerankers and SASRec) come
  from the **model cache** that Unit 14 used. Build it once, before you start, from the repository root:
  `uv run --group recsys python recsys/projects/bookrec/scripts/build_models.py`.
  Loading the cache is not cheating: it is rebuilt from the same seeded code on every CI run and checked by a
  required test. You already watched each of these models train in Units 5, 8–12 and 14.
- Use only tools you have been taught. Every cell runs in seconds.

Fill every `None` and `"TODO"` slot. Each written answer goes in the string variable it names.

## Setup — load the whole system

The cell below loads the cache and assembles the system exactly as Unit 14 did.
`system.full8` holds the eight served retrieval paths, keyed by name (the two-tower here is served through HNSW).
`system.evaluate(path)` scores any path on the val cohort and returns hit@10 **and** the beyond-accuracy readout:
`coverage`, `head_share`, `exposure_gini`, `novelty` and `diversity`.
`system.top_lists(path)` gives you the raw `{reader: top-10}` lists, and `system.readout(lists)` scores them.

It also builds `zero_train_books`: the catalog books with **no** train positive at all.
No collaborative path can ever learn about them, so how many of them your system shows is your cold-start readout.

Run it once.

In [ ]:
from bookrec import FullSystem, load_cached_models
from bookrec.system import FULL8_PATH_NAMES, POOL, FixedBlendPath

K = 10

cache = load_cached_models()          # fails with the build command if the cache is missing
system = FullSystem.from_cache(cache)
data = system.data

trained_books = {
    row["item_id"] for row in data.rows if row["split"] == "train" and row["label"] == 1
}
zero_train_books = set(data.catalog_ids) - trained_books


def cold_surfaced(lists):
    # Distinct zero-train books that appear in any reader's top-10.
    return len({book for top in lists.values() for book in top if book in zero_train_books})


def show(name, metrics):
    print(
        f"{name:<24} hit@10 {metrics['hit']:.3f}  coverage {metrics['coverage']:.3f}  "
        f"head-share {metrics['head_share']:.3f}  Gini {metrics['exposure_gini']:.3f}"
    )


print(f"{len(data.cohort)} val readers, {len(data.catalog_ids)} books, "
      f"{len(zero_train_books)} books with no train positive")
print("the eight served paths:", list(FULL8_PATH_NAMES))

## Milestone 1

### Choose your retrieval paths, and say why

A recommender is only as good as the candidates its retrieval paths propose.
Before you blend anything, measure every path **on its own**.

1. For each of the eight paths in `system.full8`, wrap it as a one-path blend,
   `FixedBlendPath({name: system.full8[name]}, pool=POOL)`, get its top-10 lists with `system.top_lists(...)`,
   score them with `system.readout(...)`, and count its cold books with `cold_surfaced(...)`.
   Store one row per path in `path_table`.
2. Choose **at least three** paths for your system in `chosen_paths`.
3. Justify **every** chosen path in one sentence in `path_reasons`, naming the number that earned its place.
   A path can earn its place by hit, by reach (coverage), or by serving cold books. Say which.

Measured reference points: the two-tower alone scores about 0.340 hit@10 at 0.177 coverage, and item-item about
0.252 at 0.490 coverage.

In [ ]:
path_table = []
for name in FULL8_PATH_NAMES:
    single = None    # TODO: a one-path FixedBlendPath over system.full8[name], pool=POOL
    lists = None     # TODO: system.top_lists(single)
    metrics = None   # TODO: system.readout(lists)
    cold = None      # TODO: cold_surfaced(lists)
    path_table.append({"path": name, "hit": None, "coverage": None, "cold_books": None})  # TODO: fill
    # TODO: print one line per path (show(name, metrics) plus its cold-book count)

chosen_paths = []    # TODO: at least three names from FULL8_PATH_NAMES

path_reasons = {
    # TODO: one sentence per chosen path, naming the measured number that earned its place, e.g.
    # "item-item": "TODO",
}

## Milestone 2

### Build the candidate pool and the blend

Turn your chosen paths into one system.
Each path proposes its top `POOL = 50` books; the blend calibrates each path's scores to `[0, 1]`, adds them with
your weights, drops the books the reader has already read, and keeps the top 10.
That is exactly the calibrated union you built in Units 6 and 10.

1. Pick a weight for each chosen path in `my_weights` (start at 1.0 each, then try at least one change).
2. Build `my_blend = FixedBlendPath({...your chosen paths...}, weights=my_weights, pool=POOL, name="my-blend")`.
3. Score it, and score the reference **equal-weight 8-path fixed blend**, `system.fixed_blend()`
   (measured: about 0.362 hit@10 at 0.206 coverage).
4. In `blend_reading`, two or three sentences: how did your blend compare with the 8-path blend on hit **and** on
   coverage, and what did changing a weight do?

In [ ]:
my_weights = {}       # TODO: {path_name: weight} for every name in chosen_paths

my_blend = None       # TODO: FixedBlendPath({n: system.full8[n] for n in chosen_paths},
                      #        weights=my_weights, pool=POOL, name="my-blend")
my_lists = None       # TODO: system.top_lists(my_blend)
my_metrics = None     # TODO: system.readout(my_lists)

fixed_blend = system.fixed_blend()
fixed_metrics = None  # TODO: system.evaluate(fixed_blend)       (about 0.362 / 0.206)

# TODO: show(...) both systems

blend_reading = "TODO"  # 2-3 sentences: hit AND coverage vs the 8-path blend; what a weight change did

## Milestone 3

### Add the learned reranker, and decide whether it stays

The last stage of a multi-stage recommender is a **learned reranker**: a small neural network that re-scores the
candidate pool from **ranking features** (each path's calibrated score, which paths proposed the book, how many
did, and four content features). It is trained with **learning to rank** on the leakage-safe Unit 11 recipe:
each reader's latest 25% of train reads are the labels, and the earlier 75% build the features.

The cache holds the full-stack reranker, `reranker-full8`, trained over all eight paths' pool.
`system.full_stack` serves it.

1. Read its training record from the cache: the first and last value of `cache.loss_history("reranker-full8")`,
   and `cache.meta("reranker-full8")["n_features"]` and `["training_rows"]`.
2. Score `system.full_stack` and count its cold books.
3. Decide: does the reranker earn its place in **your** system? Set `keep_reranker` to `True` or `False`, and set
   `final_system` and `final_lists` to the system you will ship (the reranked full stack, or your blend).
4. In `reranker_reading`, two or three sentences: what did the reranker change on hit, on coverage and on cold
   books, and why did you keep or drop it?

In [ ]:
reranker_loss = None       # TODO: cache.loss_history("reranker-full8")
first_loss = None          # TODO: reranker_loss[0]
last_loss = None           # TODO: reranker_loss[-1]
n_features = None          # TODO: cache.meta("reranker-full8")["n_features"]     (21)
training_rows = None       # TODO: cache.meta("reranker-full8")["training_rows"]

full_stack_lists = None    # TODO: system.top_lists(system.full_stack)
full_stack_metrics = None  # TODO: system.readout(full_stack_lists)
full_stack_cold = None     # TODO: cold_surfaced(full_stack_lists)

keep_reranker = None       # TODO: True or False, from YOUR numbers
final_system = None        # TODO: system.full_stack if keep_reranker else my_blend
final_lists = None         # TODO: the matching top-10 lists (full_stack_lists or my_lists)
final_metrics = None       # TODO: system.readout(final_lists)

reranker_reading = "TODO"  # 2-3 sentences: its effect on hit, coverage and cold books; keep or drop, and why

## Milestone 4

### The ablation table: what does each piece buy?

An **ablation** removes one piece at a time and measures what is lost.
Every row is a *difference* from a stated **baseline**, so always say which baseline a row is compared with.

1. **Your own ablation.** For each path in `chosen_paths`, rebuild your blend **without** that path (same weights
   for the rest), score it, and record `d_hit` and `d_coverage` = (blend without the path) − (your full blend).
   Add one **stage row** for the reranker: the full stack minus the 8-path fixed blend.
2. **The reference ablation.** `cache.readouts["ablation"]` holds Unit 14's 10-row table, measured from the same
   cache: two stage rows compared with the full stack, and eight path rows compared with the 8-path fixed blend.
   Print it with its baselines.
3. In `ablation_reading`, two or three sentences: which piece of your system matters most, which barely matters,
   and does any removal *raise* hit? (Look at the reference table too.)

In [ ]:
my_ablation = []
for dropped in chosen_paths:
    kept = None            # TODO: {n: system.full8[n] for n in chosen_paths if n != dropped}
    kept_weights = None    # TODO: the same weights, without the dropped path
    without = None         # TODO: FixedBlendPath(kept, weights=kept_weights, pool=POOL,
                           #                      name=f"minus-{dropped}")
    metrics = None         # TODO: system.evaluate(without)
    my_ablation.append({
        "row": f"minus-{dropped}",
        "baseline": "my-blend",
        "d_hit": None,       # TODO: metrics["hit"] - my_metrics["hit"]
        "d_coverage": None,  # TODO: metrics["coverage"] - my_metrics["coverage"]
    })

my_ablation.append({
    "row": "add-reranker",
    "baseline": "fixed-blend",
    "d_hit": None,       # TODO: full_stack_metrics["hit"] - fixed_metrics["hit"]
    "d_coverage": None,  # TODO: full_stack_metrics["coverage"] - fixed_metrics["coverage"]
})

reference = cache.readouts
# TODO: print reference["baselines"] (full-stack and fixed-blend hit/coverage), then every row of
#       reference["ablation"] with its baseline, d_hit and d_coverage; then print your own my_ablation rows

ablation_reading = "TODO"  # 2-3 sentences: what matters most, what barely matters, does a removal raise hit?

## Milestone 5

### Beyond accuracy, and the Part-1 benchmark

hit@10 says how often a reader gets *one* book they wanted. It says nothing about **reach** (how much of the
catalog ever appears), **exposure** (how unevenly the slots are shared between books), **novelty** or
**diversity**. Build one table that puts them side by side.

Rows, each scored with `system.evaluate(...)` (or `system.readout(...)` on lists you already have):
- **your final system** (`final_metrics`);
- the **full stack** (`full_stack_metrics`);
- the **best single path**, `system.best_single_path` (the Unit 10 hybrid, about 0.362 / 0.192);
- the **Part-1 build**, `system.part1_build()`: the Unit 6 blend of popularity, keywords, item-item and MF
  (about 0.306 / 0.333).

Columns: `hit`, `coverage`, `head_share`, `exposure_gini`, `novelty`, `diversity`, plus your cold-book count.

Then compare with the build's own table, `cache.readouts["beyond_accuracy"]`.
In `benchmark_reading`, two or three sentences: does your system beat the Part-1 build on hit? On reach? On
exposure? Does **any** system in your table beat it on every column?

In [ ]:
best_single_metrics = None   # TODO: system.evaluate(system.best_single_path)
part1_metrics = None         # TODO: system.evaluate(system.part1_build())     (about 0.306 / 0.333)

beyond_rows = [
    {"system": "my-final-system", **(final_metrics or {})},
    # TODO: add rows for the full stack, the best single path and the Part-1 build
]
columns = ["hit", "coverage", "head_share", "exposure_gini", "novelty", "diversity"]
# TODO: print the table (one line per row, every column), then print cache.readouts["beyond_accuracy"]

benchmark_reading = "TODO"  # 2-3 sentences: vs Part 1 on hit, reach, exposure; does anything dominate it?

## Milestone 6

### The design memo: defend your point on the frontier

The library director does not read notebooks. Write the memo they will read.
Each answer is two to four sentences, and **every claim cites a number from your own tables**.

- `frontier_choice`: where on the hit ↔ reach ↔ exposure frontier did you put your system, and why is that the
  right place *for a library*? Name one system that beats yours on some column, and say why you still chose yours.
- `accuracy_vs_reach`: what did you give up on reach or exposure to gain hit, or on hit to gain reach?
- `cold_start`: how many zero-train books does your system ever show? Which paths make that possible, and what
  does your choice mean for a new author's first book?
- `cost`: what does each stage cost to build and run? The cache records its build times in
  `cache.manifest["timings_s"]` (seconds per model). Which stage would you cut first to save cost, and what does
  your ablation say that would lose?
- `next_step`: one change you would test next, and the measurement that would tell you it worked.

In [ ]:
build_seconds = cache.manifest["timings_s"]
# TODO: print build_seconds, and the build time of every model your final system needs

frontier_choice = "TODO"
accuracy_vs_reach = "TODO"
cold_start = "TODO"
cost = "TODO"
next_step = "TODO"

## Make it yours

Choose one or more extensions after your six milestones are done.

- **Tune the frontier.** Try three or more weightings of your blend and list hit@10 next to coverage in one table. Add the
  Part-1 build, the full stack and your final system as rows. Which weightings are *dominated* (another one is better on
  both)?
- **A fairer shelf.** Add a small weight on a reach-friendly path (semantic or lexical) and measure what each
  0.01 of coverage costs in hit. Report the exposure Gini and head-share too.
- **Serve the cold books on purpose.** Give the feature tower or the semantic path a bigger weight and measure how
  many zero-train books appear, and at what hit cost.
- **Sequence-aware add-on.** Score the cached SASRec model on the **session log** with `system.session_top_lists()`
  (as in Unit 14), and explain why its numbers cannot be put in the same table as your main-log system.
- **Interrogate the reranker.** Refit a reranker model on the cached training matrix,
  `cache.training_matrix("reranker-full8")`, with one feature family removed (as in Unit 14's exercises), and
  explain what that family was contributing.

## Requirements checklist

**Your capstone must: justify every retrieval path with a measured number; build and score a calibrated blend;
measure the learned reranker and decide whether it stays; produce an ablation table with stated baselines; produce
a beyond-accuracy table that includes the Part-1 build; and defend your choice on the hit ↔ reach ↔ exposure
frontier in a design memo. Everything on `val` only.**

Before you hand in, check every box.

- [ ] Milestone 1: every one of the eight paths is scored alone (hit, coverage, cold books), and each chosen path
  has a one-sentence reason that cites a number.
- [ ] Milestone 2: my blend is built from my chosen paths with my weights, and is compared with the 8-path fixed
  blend on hit **and** coverage.
- [ ] Milestone 3: the learned reranker is scored, its training record is read from the cache, and I decided
  whether it stays, from my numbers.
- [ ] Milestone 4: my ablation table has one row per chosen path plus the reranker stage row, every delta is
  computed from a named baseline, and the reference 10-row table is printed.
- [ ] Milestone 5: my beyond-accuracy table has my system, the full stack, the best single path and the Part-1
  build, with all six columns.
- [ ] Milestone 6: my memo answers frontier, accuracy vs reach, cold start, cost and next step, each citing my own
  numbers.
- [ ] I never touched the sealed final split; every number is on `val`.
- [ ] The notebook runs top to bottom without errors.